# 1. PORTFOLIO DEVELOPMENT - CONSTRUCTION, VALIDATION & PRODUCTION DEFINITION

This notebook builds **one portfolio** out of the pool of approved strategy variants produced in `02_strategy_development`. Every variant that passes the robustness notebook exports a `returns.csv` into its `runs/<variant>/` folder. Here we pick the subset that really diversifies, decide how much risk each one gets, validate the combination out-of-sample and freeze the definition that goes to production.

**Reusable:** change only `PORTFOLIO` (section 1.4). Everything else comes from `configs/<PORTFOLIO>.yaml`, the single source of truth. To run it headless for another portfolio: `papermill 01_Portfolio_Construction.ipynb out.ipynb -p PORTFOLIO portfolio_2`.

### 1.1 CORE PORTFOLIO HYPOTHESIS
- **Diversification means lowering correlation, not adding systems.** Five cousins are one bet; three genuinely different systems beat fifteen look-alikes.
- **The benefit is a smoother equity curve and a smaller drawdown at similar return** (compounded return is roughly mean return minus variance / 2, so less variance also lifts compounding).
- **Correlation is not constant.** It jumps toward +1 in liquidity crashes, so diversification cuts normal risk but not tail risk. Position sizing and the risk overlay matter as much as the mix.
- **Weights must be estimated with past data only.** Estimating them on the full history and evaluating on the same history overstates what will happen live.
- **Regimes never switch a strategy on or off.** The portfolio runs everything all the time; what changes is which system carries the return in each cycle.

### 1.2 FROZEN PORTFOLIO RULES
| Rule Component | Specification |
| :--- | :--- |
| **Pool** | Approved variants: those with `runs/<variant>/returns.csv`, named `<edge>.<variant>`. Optional subset via `strategies` in the YAML |
| **Return basis** | Daily returns, net of costs, per unit of capital allocated to each variant. Same sizing criterion for every variant |
| **Selection** | Greedy by priority (YAML order): a variant enters only if its correlation with every already selected one is `<= max_corr` |
| **Weights** | `equal`, `inverse_vol`, `erc` or `hrp`, estimated on a rolling `lookback` of **past** data only |
| **Rebalance** | Every `rebalance_days`; constant weights in between |
| **Risk overlay** | Volatility targeting (scalar from trailing vol, capped by `max_leverage`) plus an optional drawdown throttle. Both use information up to the previous close |
| **Evaluation** | Walk-forward, reported only from `eval_start` (out-of-sample of the variants) |

### 1.3 PROVENANCE & REFERENCES
- **Trade It Simple** - Module 9 (Block 6, Portfolio View) and the Strategy Decorrelation Guide: correlation of returns (never prices), structural vs parametric correlation, thresholds 0.2 / 0.5, equity-curve and bad-day checks.
- **Markowitz (1952)** - Portfolio Selection. Basis of the diversification argument.
- **Maillard, Roncalli & Teiletche (2010)** - The Properties of Equally Weighted Risk Contribution Portfolios (`erc`).
- **Lopez de Prado (2016)** - Building Diversified Portfolios that Outperform Out of Sample (`hrp`).
- **Perold (1988)** - The Implementation Shortfall: the paper portfolio is not the live portfolio, so costs must already be inside the input returns.

### 1.4 PORTFOLIO TO BUILD

In [ ]:
PORTFOLIO = "portfolio_1"   # yaml name in configs/ without .yaml  (papermill parameter)

## 2. IMPORTS AND CONFIG

In [ ]:
import sys
import json
import shutil
import importlib
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ---- Paths ----
def find_upwards(name, start=None, must_be_dir=True):
    """Sube desde start (cwd por defecto) y devuelve la primera carpeta
    que contiene `name`. Devuelve esa carpeta contenedora."""
    here = (start or Path.cwd()).resolve()
    for base in (here, *here.parents):
        target = base / name
        if target.is_dir() if must_be_dir else target.exists():
            return base
    raise FileNotFoundError(f"No encontré '{name}' subiendo desde {here}")

PD_DIR = find_upwards("portfolio_development")   # 03_portfolio_development (paquete + configs + runs)
if str(PD_DIR) not in sys.path:
    sys.path.insert(0, str(PD_DIR))

from portfolio_development import config as pcfg, data as pdata, construct as pcons, metrics as pmet
from portfolio_development import backtest as pbt, analysis as pan, report as prep
for _m in (pcfg, pdata, pcons, pmet, pbt, pan, prep):
    importlib.reload(_m)   # recoge ediciones del paquete sin reiniciar el kernel (orden de dependencias)

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# ---- Portfolio config: single source of truth ----
CFG_PATH = PD_DIR / "configs" / f"{PORTFOLIO}.yaml"
cfg = pcfg.load_config(CFG_PATH)
PPY = cfg.periods_per_year
RUN_DIR = cfg.out_dir / cfg.name
RUN_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(CFG_PATH, RUN_DIR / "portfolio_config.yaml")   # copia congelada junto a los resultados

print("Imports complete.")
print(f"Portfolio: {cfg.name} | pool: {list(cfg.strategies) if cfg.strategies else 'all approved variants'}")
print(f"Selection: max_corr <= {cfg.selection.max_corr:g} | Weights: {cfg.weights.method} "
      f"(lookback {cfg.weights.lookback}d, rebalance every {cfg.weights.rebalance_days}d)")
print(f"Risk: target_vol={cfg.risk.target_vol} | max_lev={cfg.risk.max_leverage:g} | "
      f"dd_trigger={cfg.risk.dd_trigger} x{cfg.risk.dd_factor:g}")
print(f"Evaluation from: {cfg.eval_start or 'start of walk-forward'} | returns_dir: {cfg.returns_dir}")
print(f"Run folder: {RUN_DIR}")

## 3. LOAD THE CANDIDATE POOL

Each variant contributes a daily return series. Inside its own active span, days without a trade are 0.0; outside it the series is NaN (the variant did not exist yet). The coverage table shows how much of each series is actually in the market: a variant with very few active days has a noisy correlation estimate, so read every number below with that in mind.

In [ ]:
available = sorted(pdata.discover(cfg.returns_dir))
returns = pdata.load_returns(cfg.returns_dir, cfg.strategies)
assert returns.shape[1] >= 2, "A portfolio needs at least 2 variants in the pool."
assert len(returns) > cfg.weights.lookback, "Not enough history for the configured lookback."

print(f"Approved variants found under returns_dir: {len(available)}")
print("Available but NOT in this portfolio:", [a for a in available if a not in returns.columns] or "none")
display(pan.coverage(returns))

fig, ax = plt.subplots(figsize=(11, 4.5))
for c in returns:
    eq = (1 + returns[c].dropna()).cumprod()
    ax.plot(eq.index, eq.to_numpy(), lw=1, label=c)
ax.set_yscale("log")
ax.set_title("Candidate pool - equity (log, per unit of each variant's own capital)")
ax.legend(fontsize=8)
plt.show()

## 4. DEPENDENCE ANALYSIS

**Rule:** correlate **returns**, never prices or equity levels, on a common frequency. Alert above **0.50**, attention between **0.20 and 0.50**, good below **0.20**. Strongly negative values are fine, but a perfect mirror would cancel the profits too.

### 4.1 Correlation matrices
Full sample vs the evaluation window. If the two differ a lot, the diversification you measured is not stable.

In [ ]:
def heatmap(ax, m, title, vmin=-1.0, vmax=1.0, cmap="RdBu_r"):
    im = ax.imshow(m.to_numpy(), cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(m)))
    ax.set_xticklabels(m.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(m)))
    ax.set_yticklabels(m.index, fontsize=8)
    for i in range(len(m)):
        for j in range(len(m)):
            v = m.iat[i, j]
            if pd.notna(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
    ax.grid(False)
    ax.set_title(title, fontsize=10)
    return im

frames = {"Full sample": returns, "Evaluation window": returns.loc[cfg.eval_start:]}
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (title, df) in zip(axes, frames.items()):
    im = heatmap(ax, df.corr(), f"Return correlation - {title}")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

corr = returns.corr()
pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().dropna()
             .rename("corr").sort_values(ascending=False))
display(pairs.to_frame().assign(
    flag=lambda d: np.where(d["corr"] > 0.5, "RED >0.5", np.where(d["corr"] > 0.2, "attention", "ok"))))

### 4.2 Rolling correlation
Correlation is not constant. If the rolling mean or max pair correlation climbs toward 0.5 or higher in stress periods, the "diversification" disappears exactly when it is needed.

In [ ]:
rp = pan.rolling_pair_corr(returns, window=252)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(rp.index, rp.mean(axis=1).to_numpy(), lw=2, color="black", label="mean of pairs")
ax.plot(rp.index, rp.max(axis=1).to_numpy(), lw=1, color="tab:red", label="max pair")
ax.axhline(0.5, ls="--", color="tab:red", alpha=0.6)
ax.axhline(0.0, color="grey", lw=0.8)
ax.set_title("Rolling 252d pairwise correlation")
ax.legend()
plt.show()

### 4.3 Bad-day overlap
Correlation can hide shared pain. Each cell is the probability that the column variant also loses on the row variant's worst 10% losing days. Values near 1 mean the bad days coincide.

In [ ]:
ov = pan.bad_day_overlap(returns, q=0.10)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = heatmap(ax, ov, "P(column also loses | row's worst 10% losing days)", vmin=0.0, vmax=1.0, cmap="Reds")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

### 4.4 Structural rationale (fill in by hand)
For every pair that looks decorrelated, answer: **why should these two not fall together?** A structural reason (different asset, different logic, different horizon) holds under stress. A number that is low for no explainable reason is a parametric accident of the sample and tends to break in crises.

| Pair | Why they should not fall together (structural) | Verdict |
| :--- | :--- | :--- |
| | | |

## 5. SELECTION AND WEIGHTS - SNAPSHOT ON THE LATEST WINDOW

This is what the portfolio would hold **today**: the greedy correlation filter on the last `lookback` days (YAML order = priority), then each weighting method side by side. Risk contribution shows where the portfolio variance really comes from; inverse-vol only equalizes stand-alone volatility, while `erc` and `hrp` take correlations into account.

In [ ]:
window = returns.iloc[-cfg.weights.lookback:]
chosen = pcons.select(window, cfg.selection.max_corr, list(returns.columns))
assert chosen, "No variant passes the selection filter in the latest window."
dropped = [c for c in returns.columns if c not in chosen]
print(f"Latest window: {window.index[0].date()} -> {window.index[-1].date()}")
print(f"Selected ({len(chosen)}): {chosen}")
print(f"Dropped  ({len(dropped)}): {dropped}  "
      f"(incomplete in window, zero variance, or corr > {cfg.selection.max_corr:g} with a higher-priority pick)")

snap = pd.DataFrame({m: fn(window[chosen]) for m, fn in pcons.WEIGHTERS.items()})
rc = pd.DataFrame({m: pan.risk_contributions(window[chosen], snap[m]) for m in snap})
print("\nWeights by method")
display(snap)
print("Risk contribution by method")
display(rc)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
snap.plot.bar(ax=axes[0], title="Weights by method (latest window)")
rc.plot.bar(ax=axes[1], title="Risk contribution by method")
plt.tight_layout()
plt.show()

w_now = snap[cfg.weights.method].reindex(chosen)
print(f"Diversification ratio ({cfg.weights.method}): "
      f"{pan.diversification_ratio(window[chosen], w_now):.2f}  (1.00 = no diversification)")

## 6. WALK-FORWARD BACKTEST

At every rebalance the selection and weights are estimated with the previous `lookback` days only and applied to the next `rebalance_days`. The table compares the final portfolio (after the risk overlay), the portfolio without overlay, and each variant on the same evaluation window. Individual variants are unlevered and per unit of their own capital.

In [ ]:
res = pbt.run(cfg, returns)
tbl = prep.summary_table(res, PPY)
print(f"Evaluation window: {res.net.index[0].date()} -> {res.net.index[-1].date()} ({len(res.net):,} days)")
display(tbl)

n_sel = (res.weights > 0).sum(axis=1)
print(f"Rebalances: {len(res.weights)} | variants held per rebalance: min {n_sel.min()}, mean {n_sel.mean():.1f}, max {n_sel.max()}")

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
for c in res.strategies:
    e = (1 + res.strategies[c].fillna(0)).cumprod()
    ax1.plot(e.index, e.to_numpy(), lw=0.7, alpha=0.6, label=c)
eq = (1 + res.net).cumprod()
ax1.plot(eq.index, eq.to_numpy(), lw=2, color="black", label="PORTFOLIO")
ax1.set_yscale("log")
ax1.set_title("Equity - evaluation window (log)")
ax1.legend(fontsize=7, ncol=3)
ax2.fill_between(eq.index, (eq / eq.cummax() - 1).to_numpy(), 0, color="tab:red", alpha=0.5)
ax2.set_title("Portfolio drawdown")
if not res.weights.empty:
    ax3.stackplot(res.weights.index, res.weights.T.to_numpy(), labels=list(res.weights.columns))
    ax3.legend(fontsize=7, ncol=3, loc="upper left")
ax3.set_title("Weights at each rebalance (before overlay)")
plt.tight_layout()
plt.show()

## 7. RISK OVERLAY - WHAT EACH RULE CONTRIBUTES

The overlay is evaluated step by step on the same raw portfolio so we can see what volatility targeting and the drawdown throttle each buy. The overlay is computed on the full history and then cut to the evaluation window, so its volatility estimate is already warmed up when the window starts.

In [ ]:
raw_full = pbt.run(replace(cfg, eval_start=None), returns).raw
ablation = {
    "no overlay": replace(cfg.risk, target_vol=None, dd_trigger=None),
    "vol targeting only": replace(cfg.risk, dd_trigger=None),
    "vol targeting + dd throttle (config)": cfg.risk,
}
overlay_ret = {k: pbt.apply_risk(raw_full, v, PPY).loc[cfg.eval_start:] for k, v in ablation.items()}
display(pd.DataFrame({k: pmet.summary(v, PPY) for k, v in overlay_ret.items()}).T)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for k, v in overlay_ret.items():
    e = (1 + v).cumprod()
    ax1.plot(e.index, e.to_numpy(), lw=1.3, label=k)
    ax2.plot(e.index, (e / e.cummax() - 1).to_numpy(), lw=1, label=k)
ax1.set_yscale("log")
ax1.set_title("Equity by overlay step (log)")
ax1.legend(fontsize=8)
ax2.set_title("Drawdown by overlay step")
plt.tight_layout()
plt.show()

## 8. ROBUSTNESS

A portfolio that only works with one weighting method, one lookback or one lucky variant is a backtest artefact. Each test below tries to break the result.

### 8.1 Weighting methods
Same pool, same selection, same overlay; only the weights change. If the four methods tell the same story the result does not depend on the method.

In [ ]:
comparison = pbt.compare_methods(cfg, returns)
display(comparison)

### 8.2 Leave-one-out
Remove one variant at a time. If removing any single variant destroys the portfolio, you are carrying hidden concentration.

In [ ]:
loo = pan.leave_one_out(cfg, returns)
display(loo)

### 8.3 Portfolio-level parameter sensitivity
Sharpe and Calmar over a grid of `lookback` x `max_corr`. We want a broad plateau, not a single good cell. Cells that need more history than available show as NaN.

In [ ]:
rows = []
for lb in (126, 252, 504):
    for mc in (0.3, 0.5, 0.7):
        c = replace(cfg, weights=replace(cfg.weights, lookback=lb), selection=replace(cfg.selection, max_corr=mc))
        try:
            s = pmet.summary(pbt.run(c, returns).net, PPY)
        except ValueError:   # not enough history for this lookback
            s = {"sharpe": np.nan, "calmar": np.nan}
        rows.append({"lookback": lb, "max_corr": mc, **s})
grid = pd.DataFrame(rows)
print("Sharpe")
display(grid.pivot(index="lookback", columns="max_corr", values="sharpe"))
print("Calmar")
display(grid.pivot(index="lookback", columns="max_corr", values="calmar"))

### 8.4 Year-by-year stability and worst windows
Calendar-year returns of the portfolio against each variant, and the five worst non-overlapping 21-day windows with what each variant did in them. The point is to see whether the diversification showed up when it hurt.

In [ ]:
yearly = pan.yearly_returns(pd.concat([res.net.rename("PORTFOLIO"), res.strategies], axis=1))
display(yearly)

worst = pan.worst_windows(res.net, res.strategies, window=21, n=5)
display(worst)

## 9. DIVERSIFICATION SCORECARD

Acceptance criteria, fixed before looking at the results. The portfolio is confirmed only if **all** of them pass; anything else means going back to selection, weights or the pool.

In [ ]:
oos = res.strategies
indiv = pd.DataFrame({c: pmet.summary(oos[c], PPY) for c in oos}).T
port = pmet.summary(res.net, PPY)
n = len(chosen)

cm = oos[chosen].corr().to_numpy()
avg_corr = float(cm[np.triu_indices_from(cm, k=1)].mean()) if n > 1 else 0.0
max_rc = float(rc[cfg.weights.method].max())

checks = {
    f"avg pairwise corr (evaluation window) <= {cfg.selection.max_corr:g}": bool(avg_corr <= cfg.selection.max_corr),
    "portfolio max drawdown better than median variant": bool(port["max_dd"] > indiv["max_dd"].median()),
    "portfolio Calmar >= median variant Calmar": bool(port["calmar"] >= indiv["calmar"].median()),
    "max risk contribution <= 2x equal share": bool(max_rc <= 2.0 / n),
    "all weighting methods have Sharpe > 0": bool((comparison["sharpe"] > 0).all()),
    "leave-one-out keeps Sharpe > 0": bool((loo["sharpe"] > 0).all()),
}
confirmed = all(checks.values())

print(f"Average pairwise correlation (selected, evaluation window): {avg_corr:.2f}")
print(f"Max risk contribution: {max_rc:.0%} (equal share would be {1 / n:.0%})")
display(pd.Series(checks, name="pass").to_frame())
print("PORTFOLIO CONFIRMED" if confirmed else "PORTFOLIO NOT CONFIRMED - review the failed checks")

## 10. SAVE THE RESULTS OF THIS NOTEBOOK FOR THE PORTFOLIO

Two JSON files are written. `01_portfolio.json` is the audit trail (what was tested and whether it passed). `portfolio_definition.json` is what production reads: selected variants, target weights **before** the overlay (the volatility-targeting scalar is applied on top at run time), rebalance frequency and risk parameters.

In [ ]:
out, _ = prep.save_report(res, cfg, comparison)   # summary.csv, weights.csv, corr.csv, returns.csv, comparison.csv, report.png
loo.to_csv(RUN_DIR / "leave_one_out.csv")
yearly.to_csv(RUN_DIR / "yearly_returns.csv")

prep.save_json(RUN_DIR, "portfolio_definition.json", {
    "portfolio": cfg.name,
    "method": cfg.weights.method,
    "target_weights": {k: float(v) for k, v in w_now.items()},
    "estimation_window": [str(window.index[0].date()), str(window.index[-1].date())],
    "rebalance_days": cfg.weights.rebalance_days,
    "risk": asdict(cfg.risk),
    "periods_per_year": PPY,
})
prep.save_json(RUN_DIR, "01_portfolio.json", {
    "portfolio": cfg.name,
    "pool": list(returns.columns),
    "selected_latest": chosen,
    "evaluation_window": [str(res.net.index[0].date()), str(res.net.index[-1].date())],
    "portfolio_metrics": {k: float(v) for k, v in port.items()},
    "median_variant_metrics": {k: float(v) for k, v in indiv.median().items()},
    "avg_pairwise_corr": avg_corr,
    "max_risk_contribution": max_rc,
    "checks": checks,
    "portfolio_confirmed": bool(confirmed),
})
print(f"Saved to: {RUN_DIR}")